# Suite STO — Stores

A store is the root object of a body of data: it holds schemas by name and the instances of its singleton schemas, and
its data is what those reach; everything that looks a schema up takes one. `Proxies.OfStore` and `Bindings.OfStore`
implement the protocol `Stores.Store`. Stores are isolated from one another; objects move between them as
snapshots.

In [ ]:
import { Bindings as Bd, JSON, Modules, Proxies, Reflection, Schemas as S, Stores, Validators } from "@mbse/schemas/Framework";
import { AttributeError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, conformance_problems, instance_problems, raises, same, text } from "./support.js";

let Item = new S.OfObject.Builder().name("Item").ref().properties(text("name")).create();
const Holds = new S.OfRelation.Builder().name("Holds").links("owner", "item").properties(text("label")).create();
const Shelved = new S.OfRelation.Builder().name("Shelved").links("shelf", "box").create();
const Box = new S.OfObject.Builder().name("Box").ref().properties(text("name")).relations((r) => r.name("items").of(Holds).me("owner"),
  (r) => r.name("shelves").of(Shelved).me("box")).create();
const Shelf = new S.OfObject.Builder().name("Shelf").ref().singleton("the.Shelf").properties(text("name")).relations(
  (r) => r.name("boxes").of(Shelved).me("shelf")).create();
const Annex = new S.OfObject.Builder().name("Annex").ref().singleton("the.Annex").relations((r) => r.name("boxes").of(Shelved).me("shelf")).create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("boxes").of(Holds).me("item")).update();
const Port = new S.OfObject.Builder().properties(text("name")).relations((r) => r.name("wired").of(Holds).me("owner")).create();
const Card = new S.OfObject.Builder().name("Card").ref().properties((p) => p.name("port").of(Port)).create();
const SCHEMAS = new Map<string, S.OfObject.Data | S.OfRelation.Data>(
  [Item, Box, Holds, Shelf, Shelved, Annex, Card].map((schema) => [schema.name as string, schema]));

/** A proxy store with the schemas named, of Item, Box, Holds, Shelf, Shelved, Annex and Card. */
function stocked(...names: string[]): Proxies.OfStore {
  const store = new Proxies.OfStore();
  for (const name of names) store.register(SCHEMAS.get(name) as S.OfObject.Data);
  return store;
}

## STO-01 · A store holds schemas by name

In [ ]:
{
  const store = stocked("Item", "Box", "Holds");
  assert(same(store.names(), ["Schemas.Module", "Item", "Box", "Holds"])); // every proxy store starts with the meta-schemas
  assert(store.schema("Box") === Box && store.registered("Holds") === Holds && store.name_of(Item) === "Item");
  raises(ValueError, () => store.register(Box), "schema 'Box' is already registered");
  raises(ValueError, () => store.register(new S.OfObject.Builder().ref().create()),
    "a schema needs a name to be registered; name it with its builder's .name()");
  raises(AttributeError, () => store.schema("Nope"), "no schema registered as 'Nope'");
  raises(TypeError, () => store.schema("Holds"), "'Holds' is a relation; no relation builder is exposed");
  raises(LookupError, () => store.registered("Nope"), "no schema registered as 'Nope'");
  raises(LookupError, () => store.name_of(Port), "schema is not registered");
  store.register(new S.OfObject.Builder(Item).name("register").clone()); // a name a method shadows: through builder()
  assert(store.builder("register").name("r").create().schema_name() === "register");
  raises(AttributeError, () => store["_hidden"], "_hidden");
  assert(store.Box().name("b").create().name === "b" && store.builder("Box").name("c").create().name === "c");
}

## STO-02 · A store's data is what its singletons reach; the rest is transient

In [ ]:
{
  const STOCK = ["Item", "Box", "Holds", "Shelf", "Shelved", "Card"];
  const store = stocked(...STOCK);
  const shelf = store.singleton("the.Shelf"); // created with its schema
  assert(same(store.extent("Shelf"), [shelf]) && store.extent("Box").length === 0);
  const a = store.Item().name("a").create();
  const box = store.Box().name("box").items((e: any) => e.item(a).label("first")).create();
  assert(store.extent("Box").length === 0 && store.extent("Item").length === 0); // nothing reachable from a singleton links them
  store.Shelf(shelf).boxes((e: any) => e.box(box)).update();
  assert(same(store.extent("Box"), [box]) && same(store.extent("Item"), [a])); // reachable now, transitively
  store.Box(box).items((e: any) => e.item((i: any) => i.name("inline"))).update();
  assert(same(store.extent("Item").map((i) => i.name), ["a", "inline"]));
  store.register(Annex); // a second root, reaching the same box
  store.Annex(store.singleton("the.Annex")).boxes((e: any) => e.box(box)).update();
  assert(same(store.extent("Box"), [box]) && same(store.extent("Annex"), [store.singleton("the.Annex")]));
  store.Card().port((p: any) => p.name("p1")).create();
  assert(store.extent("Card").length === 0);

  // There is one instance of a singleton schema per store; decoding one updates it.
  raises(ValueError, () => store.Shelf().create(), "create() would make a second 'Shelf'; its one instance is store.singleton('the.Shelf')");
  raises(ValueError, () => store.Shelf(shelf).clone(), "clone() would make a second 'Shelf'");
  raises(ValueError, () => store.register(new S.OfObject.Builder(Shelf).name("Shelf2").clone()),
    "singleton 'the.Shelf' is already registered");
  raises(LookupError, () => store.singleton("nope"), "no singleton named 'nope'");
  const other = stocked(...STOCK, "Annex");
  const copy = JSON.FromJSON(other).Reachable(Shelf, JSON.ToJSON(store).Reachable(Shelf, shelf));
  assert(copy === other.singleton("the.Shelf") && same(other.extent("Box").map((b) => b.name), ["box"]));
  assert(same(other.extent("Annex"), [other.singleton("the.Annex")])); // the annex in the snapshot updated other's own
  raises(AttributeError, () => store.extent("Nope"), "no schema registered as 'Nope'");
  raises(TypeError, () => store.extent("Holds"), "'Holds' is a relation");
}

## STO-03 · Stores are isolated; objects move between them as snapshots

In [ ]:
{
  const [one, two] = [stocked("Item", "Box", "Holds"), stocked("Item", "Box", "Holds")];
  const Other = new S.OfObject.Builder().name("Box").ref().properties(text("title")).create();
  const three = new Proxies.OfStore();
  three.register(Other); // the same name, another schema
  assert(three.Box().title("t").create().title === "t");
  const x = one.Item().name("x").create();
  const y = two.Item().name("y").create();
  raises(TypeError, () => one.Box().items((e: any) => e.item(y)), "the object belongs to another store");
  raises(TypeError, () => two.Item(x), "the object belongs to another store");
  const b = one.Box().name("b").items((e: any) => e.item(x).label("l")).create();
  assert(Proxies.store_of(b) === one && Proxies.store_of(b) !== two);
  const moved = JSON.FromJSON(two).Reachable(Box, JSON.ToJSON(one).Reachable(Box, b)) as Proxies.Instance;
  assert(Proxies.store_of(moved) === two && moved !== b && moved.name === "b");
  assert(Validators.Validate(two).Reachable(Box, moved).length === 0);
  assert(same(Validators.Validate(three)(Box, b), ["the value is a 'Box', not an instance of the given schema"]));
  const card = stocked("Card").Card().port((p: any) => p.name("p")).create();
  assert(Proxies.store_of(card.port) === Proxies.store_of(card)); // a value object's store is its owner's
  raises(TypeError, () => Proxies.store_of("text"), "the value does not belong to a store");
  const module = Modules.module(one, [Item]) as Proxies.Instance;
  assert(Proxies.store_of(module) === one);
}

## STO-04 · Every implementation meets the protocol

In [ ]:
{
  assert(conformance_problems(Proxies.OfStore, "Store").length === 0);
  assert(conformance_problems(Bd.OfStore, "Store").length === 0);
  assert(instance_problems(new Proxies.OfStore(), "Store").length === 0);

  /** A class of the program's own, bound to Item or to a singleton schema, One. */
  class Thing {
    constructor(public binding: Bd.Binding, public name: string | null = null) {}

    identity(): unknown {
      return this;
    }

    schema_name(): string {
      return this.binding === ONE_BINDING ? "One" : "Item";
    }

    owner(): null {
      return null;
    }

    accept(visitor: any): void {
      Bd.accept(this.binding, this, visitor);
    }
  }

  const nameOf = (s: Bd.State) => (s.values.get("name") as string | undefined) ?? null;
  const THING = new Bd.Binding(Item, (t: Thing) => new Bd.State(new Map([["name", t.name]])), (s) => new Thing(THING, nameOf(s)),
    (t: Thing, s) => Object.assign(t, { name: nameOf(s) }), { implied: ["boxes"] });
  const bound = new Bd.OfStore([[Item, (instance?: Thing) => new Bd.Builder(THING, instance)]], [Holds]);
  assert(instance_problems(bound, "Store").length === 0 && bound.extent("Item").length === 0);
  assert((bound as any).then === undefined); // JavaScript's probes are not schema lookups
  assert(same(bound.names(), ["Item", "Holds"])); // a store of bound classes holds no meta-schemas
  const One = new S.OfObject.Builder().name("One").ref().singleton("the.One").properties(text("name")).create();
  const ONE_BINDING: Bd.Binding = new Bd.Binding(One, (t: Thing) => new Bd.State(new Map([["name", t.name]])),
    (s) => new Thing(ONE_BINDING, nameOf(s)));
  const ones = new Bd.OfStore([[One, (instance?: Thing) => new Bd.Builder(ONE_BINDING, instance)]]);
  assert(ones.singleton("the.One") instanceof Thing && same(ones.extent("One"), [ones.singleton("the.One")]));
  const first = (bound as any).Item().property("name", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("first")))).create();
  assert(bound.builder("Item", first).clone() !== first && bound.builder("Item", first).update() === first);
  assert(bound.member(first, "name") === "first" && bound.extent("Item").length === 0); // no singleton reaches it
  assert((JSON.FromJSON(bound)(Item, JSON.ToJSON(bound)(Item, first)) as Thing).name === "first");
  raises(AttributeError, () => (bound as any).Nope, "Nope");
  raises(TypeError, () => bound.builder("Holds"), "'Holds' is a relation");
  raises(TypeError, () => bound.extent("Holds"), "'Holds' is a relation");
}

## STO-05 · A value object not yet placed belongs to no store

In [ ]:
{
  const P = Proxies._internals;
  const builder = new P.RecordBuilderTarget(Port);
  builder.property("name", (p) => p.value((a) => a.as_native((n) => n.set("loose"))));
  const loose = builder.build(); // built, but not placed in an owner
  const seen: string[] = [];
  loose.accept({ identify: () => undefined, property: (n: string) => seen.push(n), adjacency: (n: string) => seen.push(n) } as any);
  assert(same(seen, ["name"]) && loose.schema_name() === ""); // its properties, and no entries; its schema is named nowhere
  raises(TypeError, () => Proxies.store_of(loose), "the value does not belong to a store");
  raises(TypeError, () => new P.RecordBuilderTarget(Port).adjacency("wired", (a) => a.add((e: any) => e.item((i: any) => i.name("new")))),
    "an object is created through a link only by a store's builder");
}

## STO-06 · Stores combined: each name its store's, the singletons all of theirs, extents reaching across them

In [ ]:
{
  const Points = new S.OfRelation.Builder().name("Points").links("index", "target").create();
  const IndexSchema = new S.OfObject.Builder().name("Index").ref().singleton("Index").relations(
    (r) => r.name("targets").of(Points).me("index")).create();

  /** A root of the program's own, which links to schemas held by another store. */
  class Index {
    targets: unknown[];

    constructor(targets: Iterable<unknown> = []) {
      this.targets = [...targets];
    }

    identity(): unknown {
      return "index";
    }

    schema_name(): string {
      return "Index";
    }

    owner(): null {
      return null;
    }

    accept(visitor: any): void {
      Bd.accept(INDEXED, this, visitor);
    }
  }

  const INDEXED = new Bd.Binding(IndexSchema, (index: Index) => new Bd.State(new Map(), new Map([["targets",
    index.targets.map((t) => new Bd.Entry(new Map([["target", t]])))]])),
    (state: Bd.State) => new Index((state.entries.get("targets") ?? []).map((e) => e.links.get("target"))));
  const Phone = new S.OfObject.Builder().name("Phone").properties(text("number")).create();
  const Card = new S.OfObject.Builder().name("Card").ref().properties(text("name")).create();
  const listed = new Proxies.OfStore();
  listed.register(Phone);
  const schemas = Reflection.of(listed);
  const indexes = new Bd.OfStore([[IndexSchema, (instance?: Index) => new Bd.Builder(INDEXED, instance)]], [Points]);
  (indexes.singleton("Index") as unknown as Index).targets = [Card, Phone]; // a schema the other store does not list, and one it does
  const both = new Stores.Combined(schemas, indexes);
  assert(same(both.names(), [...schemas.names(), "Index", "Points"]) && same(both.stores, [schemas, indexes]));
  assert(same(both.extent("Schemas.Object"), [Phone, Card])); // the owning store's, then what the singletons reach
  assert(same(both.extent("Index"), [indexes.singleton("Index")]) && both.singleton("Index") === indexes.singleton("Index"));
  assert(both.schema("Index") === IndexSchema && both.registered("Points") === Points && both.name_of(Points) === "Points");
  assert(both.member(Card, "name") === "Card" && both.builder("Index") instanceof Bd.Builder);
  raises(AttributeError, () => both.schema("Nope"), "no schema registered as 'Nope'");
  raises(LookupError, () => both.registered("Nope"), "no schema registered as 'Nope'");
  raises(LookupError, () => both.name_of(Phone), "schema is not registered");
  raises(LookupError, () => both.singleton("Nope"), "no singleton named 'Nope'");
  raises(ValueError, () => new Stores.Combined(schemas, Reflection.of(new Proxies.OfStore())), "schema 'Schemas.Native' is registered by two of the stores");
  const nested = new Stores.Combined(new Stores.Combined(schemas), indexes, { names: () => [] } as never); // and one with no singletons
  assert(same(nested.extent("Schemas.Object"), [Phone, Card]) && nested.singleton("Index") === indexes.singleton("Index"));
}